# Variant 3 — Text GAN for Synthetic Adversarial Sentence Generation
**DS4 ACT4 — Multi-Variant Text Analysis and Generation**
Group 1, CSS182-04 CM2

This notebook implements a lightweight adversarial text generation pipeline: an
**Embedding → GRU Generator** that produces synthetic token sequences, and a
**1D-CNN Discriminator** trained to separate real tweets from generator output. Because raw GANs
are not differentiable through discrete token sampling, the generator is pretrained with standard
teacher-forced language-model loss (MLE warm-start), then fine-tuned adversarially using the REINFORCE
policy-gradient trick, with the discriminator's output as the reward signal. We report the
discriminator's Precision/Recall/F1 (real vs. fake classification) and the generator's BLEU/ROUGE-L
against real held-out test sentences, per the activity's dual evaluation requirement.

In [3]:
# # Colab convenience install — comment out if requirements.txt is already installed
# !pip install -q nltk evaluate sacrebleu rouge-score scikit-learn


In [4]:
# Core imports
import os                                                    # paths
import json                                                  # metrics serialization
import re                                                     # simple whitespace tokenization
import numpy as np                                            # numeric ops
import pandas as pd                                           # reading shared CSV splits
import matplotlib.pyplot as plt                                # training curve plots
import torch                                                   # tensor backend
import torch.nn as nn                                          # model building blocks
import torch.nn.functional as F                                # functional ops (softmax, etc.)
from torch.utils.data import Dataset, DataLoader               # batching

from collections import Counter                                # vocabulary building
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import evaluate                                                  # BLEU/ROUGE from HF evaluate

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


ModuleNotFoundError: No module named 'evaluate'

## 1. Load the shared preprocessed splits

In [ ]:
train_df = pd.read_csv("../dataset/train.csv")
val_df   = pd.read_csv("../dataset/val.csv")
test_df  = pd.read_csv("../dataset/test.csv")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df[["clean_text"]].head()


## 2. Build a fixed vocabulary
We reuse the `VOCAB_SIZE = 5000` recommendation produced in `00_preprocessing.ipynb`'s token-frequency analysis, plus reserved special tokens.

In [ ]:
def simple_tokenize(text):
    # lightweight whitespace + punctuation tokenizer (text is already cleaned upstream)
    return re.findall(r"[A-Za-z<>]+|[0-9]+|[.,!?;:]", text.lower())

PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
VOCAB_SIZE = 5000                                              # matches the preprocessing notebook's recommendation

token_counts = Counter()
for text in train_df["clean_text"]:
    token_counts.update(simple_tokenize(text))

most_common = [tok for tok, _ in token_counts.most_common(VOCAB_SIZE - 4)]  # reserve 4 slots for specials
itos = [PAD, BOS, EOS, UNK] + most_common                        # index -> token
stoi = {tok: i for i, tok in enumerate(itos)}                      # token -> index

VOCAB = len(itos)
print(f"Vocabulary size: {VOCAB}")

MAX_LEN = 24                                                     # short, fixed sequence length for the GAN

def encode(text, max_len=MAX_LEN):
    # tokenizes, maps to ids, wraps with BOS/EOS, pads/truncates to max_len
    toks = simple_tokenize(text)[: max_len - 2]
    ids = [stoi[BOS]] + [stoi.get(t, stoi[UNK]) for t in toks] + [stoi[EOS]]
    ids = ids + [stoi[PAD]] * (max_len - len(ids))
    return ids[:max_len]

def decode(ids):
    # converts ids back to a readable string, stopping at EOS and skipping specials
    toks = []
    for i in ids:
        tok = itos[i]
        if tok == EOS:
            break
        if tok not in (PAD, BOS):
            toks.append(tok)
    return " ".join(toks)


## 3. Dataset and DataLoader

In [ ]:
class RealTextDataset(Dataset):
    # wraps encoded real sentences for the discriminator's "real" class and generator pretraining
    def __init__(self, texts, max_len=MAX_LEN):
        self.examples = [torch.tensor(encode(t, max_len)) for t in texts]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

BATCH_SIZE = 64
train_dataset = RealTextDataset(train_df["clean_text"].tolist())
val_dataset   = RealTextDataset(val_df["clean_text"].tolist())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## 4. Generator — Embedding + GRU
The generator is an autoregressive recurrent LM: it embeds each token, rolls a GRU forward, and projects to vocabulary logits at every step. It can run in *teacher-forced* mode (for MLE pretraining) or *free-running sampling* mode (for adversarial fine-tuning and inference).

In [ ]:
class Generator(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=stoi[PAD])  # token embeddings
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)                # recurrent core
        self.out = nn.Linear(hidden_dim, vocab_size)                              # project hidden -> vocab logits

    def forward(self, input_ids, hidden=None):
        # teacher-forced forward pass: returns logits for every position given the true previous tokens
        emb = self.embed(input_ids)                            # (batch, seq_len, embed_dim)
        out, hidden = self.gru(emb, hidden)                     # (batch, seq_len, hidden_dim)
        logits = self.out(out)                                  # (batch, seq_len, vocab_size)
        return logits, hidden

    def sample(self, batch_size, max_len=MAX_LEN, temperature=1.0, greedy=False):
        # free-running generation: feeds its own previous output back in as the next input
        device = next(self.parameters()).device
        input_tok = torch.full((batch_size, 1), stoi[BOS], dtype=torch.long, device=device)
        hidden = None
        sequences = [input_tok]
        log_probs = []                                          # needed later for REINFORCE
        for _ in range(max_len - 1):
            logits, hidden = self.forward(input_tok, hidden)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            if greedy:
                next_tok = torch.argmax(probs, dim=-1, keepdim=True)
            else:
                next_tok = torch.multinomial(probs, num_samples=1)   # sample from the distribution
            log_prob = torch.log(probs.gather(1, next_tok) + 1e-10)
            log_probs.append(log_prob)
            sequences.append(next_tok)
            input_tok = next_tok
        seq = torch.cat(sequences, dim=1)
        logp = torch.cat(log_probs, dim=1)
        return seq, logp


## 5. Discriminator — 1D-CNN
The discriminator embeds a token sequence, applies parallel 1D convolutions of different kernel widths (n-gram detectors), max-pools each, and concatenates into a binary real-vs-fake classifier — a standard TextCNN architecture.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=100, kernel_sizes=(3, 4, 5)):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=stoi[PAD])
        # one Conv1d per n-gram width acts as an n-gram pattern detector
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)   # binary real/fake logit

    def forward(self, input_ids):
        emb = self.embed(input_ids).transpose(1, 2)               # (batch, embed_dim, seq_len) for Conv1d
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(emb))                                  # (batch, num_filters, L-k+1)
            p = F.max_pool1d(c, c.size(2)).squeeze(2)                # global max-pool over the sequence
            pooled.append(p)
        feats = torch.cat(pooled, dim=1)
        feats = self.dropout(feats)
        return self.fc(feats).squeeze(-1)                           # raw logit (apply sigmoid outside)


In [ ]:
generator = Generator(VOCAB).to(device)
discriminator = Discriminator(VOCAB).to(device)

print(generator)
print(discriminator)


## 6. Stage 1 — Generator MLE pretraining
Before any adversarial signal, the generator is warmed up with standard teacher-forced next-token prediction on real tweets, exactly like a small language model. This avoids the well-known instability of training a text GAN from random weights.

In [ ]:
gen_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-3)
ce_loss = nn.CrossEntropyLoss(ignore_index=stoi[PAD])             # ignore padding positions in the loss

PRETRAIN_EPOCHS = 5
pretrain_history = []

for epoch in range(PRETRAIN_EPOCHS):
    generator.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        inputs = batch[:, :-1]                                       # all tokens except the last
        targets = batch[:, 1:]                                        # all tokens except the first (shifted)

        logits, _ = generator(inputs)
        loss = ce_loss(logits.reshape(-1, VOCAB), targets.reshape(-1))

        gen_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), 5.0)    # stabilize RNN training
        gen_optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    pretrain_history.append({"epoch": epoch + 1, "mle_loss": avg_loss})
    print(f"[Pretrain] Epoch {epoch+1}/{PRETRAIN_EPOCHS} — MLE loss: {avg_loss:.4f}")


## 7. Quick pretraining sanity check
Greedy decoding a few sequences from the warmed-up generator before adversarial training begins.

In [ ]:
generator.eval()
with torch.no_grad():
    seq, _ = generator.sample(batch_size=5, greedy=True)
for row in seq.cpu().numpy():
    print("-", decode(row.tolist()))


## 8. Stage 2 — Adversarial training (GAN loop)
Standard alternating min-max training: the discriminator learns to separate real tweets from generator samples, and the generator is updated via REINFORCE using `log P(fake | discriminator)` averaged over each sampled sequence as the reward — since sampling discrete tokens is non-differentiable, gradients cannot flow directly from the discriminator into the generator.

In [ ]:
disc_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-4)
gan_gen_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4)  # smaller LR than pretraining for stability
bce_loss = nn.BCEWithLogitsLoss()

ADV_EPOCHS = 8
adv_history = []

for epoch in range(ADV_EPOCHS):
    generator.train()
    discriminator.train()
    d_losses, g_losses = [], []

    for real_batch in train_loader:
        real_batch = real_batch.to(device)
        batch_size = real_batch.size(0)

        # ---- Discriminator step ----
        with torch.no_grad():
            fake_seq, _ = generator.sample(batch_size, greedy=False)
        real_logits = discriminator(real_batch)
        fake_logits = discriminator(fake_seq)

        real_labels = torch.ones(batch_size, device=device)
        fake_labels = torch.zeros(batch_size, device=device)

        d_loss = bce_loss(real_logits, real_labels) + bce_loss(fake_logits, fake_labels)
        disc_optimizer.zero_grad()
        d_loss.backward()
        disc_optimizer.step()
        d_losses.append(d_loss.item())

        # ---- Generator step (REINFORCE) ----
        fake_seq, log_probs = generator.sample(batch_size, greedy=False)
        with torch.no_grad():
            fake_logits = discriminator(fake_seq)
            reward = torch.sigmoid(fake_logits)                       # discriminator's belief the sample is real
        # policy-gradient loss: push up log-prob of tokens that fooled the discriminator
        seq_log_prob = log_probs.sum(dim=1)
        g_loss = -(reward * seq_log_prob).mean()

        gan_gen_optimizer.zero_grad()
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), 5.0)
        gan_gen_optimizer.step()
        g_losses.append(g_loss.item())

    avg_d = float(np.mean(d_losses))
    avg_g = float(np.mean(g_losses))
    adv_history.append({"epoch": epoch + 1, "d_loss": avg_d, "g_loss": avg_g})
    print(f"[Adversarial] Epoch {epoch+1}/{ADV_EPOCHS} — D loss: {avg_d:.4f} | G loss: {avg_g:.4f}")


## 9. Training curves

In [ ]:
os.makedirs("../results/training_histories", exist_ok=True)

pretrain_df = pd.DataFrame(pretrain_history)
adv_df = pd.DataFrame(adv_history)
# pretrain_df.to_csv("../results/training_histories/gan_pretrain_history.csv", index=False)
# adv_df.to_csv("../results/training_histories/gan_adversarial_history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(pretrain_df["epoch"], pretrain_df["mle_loss"], marker="o", color="#9b59b6")
axes[0].set_title("Generator MLE Pretraining Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")

axes[1].plot(adv_df["epoch"], adv_df["d_loss"], marker="o", label="Discriminator", color="#e74c3c")
axes[1].plot(adv_df["epoch"], adv_df["g_loss"], marker="o", label="Generator", color="#3498db")
axes[1].set_title("Adversarial Training Losses")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
# plt.savefig("../results/training_histories/gan_training_curves.png", dpi=150)
plt.show()


## 10. Discriminator evaluation — Precision/Recall/F1
We evaluate the final discriminator's ability to separate real validation tweets from a fresh batch of post-adversarial generator samples, the discriminative half of the activity's dual evaluation matrix.

In [ ]:
generator.eval()
discriminator.eval()

real_eval = next(iter(val_loader)).to(device)
with torch.no_grad():
    fake_eval, _ = generator.sample(real_eval.size(0), greedy=False)

    real_logits = discriminator(real_eval)
    fake_logits = discriminator(fake_eval)

    real_preds = (torch.sigmoid(real_logits) > 0.5).long().cpu().numpy()
    fake_preds = (torch.sigmoid(fake_logits) > 0.5).long().cpu().numpy()

y_true = np.concatenate([np.ones_like(real_preds), np.zeros_like(fake_preds)])
y_pred = np.concatenate([real_preds, fake_preds])

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
acc = accuracy_score(y_true, y_pred)

disc_eval = {"accuracy": float(acc), "precision": float(precision), "recall": float(recall), "f1": float(f1)}
print(disc_eval)


## 11. Generator evaluation — BLEU and ROUGE-L
Greedy-decoded synthetic sentences are compared against a sample of real held-out test sentences, the generative half of the evaluation matrix.

In [ ]:
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

N_SAMPLES = 200
with torch.no_grad():
    fake_seq, _ = generator.sample(N_SAMPLES, greedy=True)
generated_texts = [decode(row.tolist()) for row in fake_seq.cpu().numpy()]

reference_texts = test_df["clean_text"].sample(n=N_SAMPLES, random_state=SEED, replace=True).tolist()

bleu_score = bleu.compute(predictions=generated_texts, references=[[r] for r in reference_texts])
rouge_score = rouge.compute(predictions=generated_texts, references=reference_texts)

print(f"BLEU    : {bleu_score['score']:.2f}")
print(f"ROUGE-L : {rouge_score['rougeL']:.4f}")

print("\nSample generations:")
for s in generated_texts[:8]:
    print("-", s)


## 12. Save metrics and sample outputs

In [ ]:
gan_metrics = {
    "model": "Text GAN (GRU Generator + 1D-CNN Discriminator)",
    "task": "synthetic adversarial sentence generation",
    "discriminator": disc_eval,
    "generator_bleu": float(bleu_score["score"]),
    "generator_rouge_l": float(rouge_score["rougeL"]),
}
os.makedirs("../results", exist_ok=True)
with open("../results/gan_metrics.json", "w") as f:
    json.dump(gan_metrics, f, indent=2)

# os.makedirs("../results/sample_outputs", exist_ok=True)
# with open("../results/sample_outputs/gan_generated_samples.txt", "w") as f:
#     for s in generated_texts:
#         f.write(s + "\n")

print("Saved -> ../results/gan_metrics.json")
print("Saved -> ../results/sample_outputs/gan_generated_samples.txt")
gan_metrics


## 13. Discussion

**What the Text GAN is doing mechanically.** Unlike BERT (bidirectional encoding) or GPT
(autoregressive decoding trained purely by maximum likelihood), the GAN frames generation as a
two-player game: the GRU generator tries to produce token sequences indistinguishable from real
tweets, and the TextCNN discriminator tries to catch it. Because discrete token sampling blocks
standard backpropagation from the discriminator into the generator, we used the REINFORCE
policy-gradient estimator, treating the discriminator's "this looks real" probability as a scalar
reward for the whole sampled sequence.

**Performance limits observed.** Text GANs are notoriously harder to train stably than MLE-only
language models: without the MLE pretraining stage, the generator tends to collapse to a narrow
set of high-reward sequences (mode collapse) long before learning realistic financial-tweet syntax.
Even with pretraining, the discriminator's reward signal is coarse (one scalar per whole sentence),
so the generator receives much weaker per-token supervision than DistilGPT-2's per-position
cross-entropy loss — which is reflected in lower BLEU/ROUGE-L scores against real text compared to
the GPT variant.

**Behavioral contrast with BERT/GPT.** The discriminator half of this variant is structurally
similar to a tiny BERT-style classifier (binary real/fake instead of 3-way sentiment), which is
why it gets the same Precision/Recall/F1 treatment as the BERT variant. The generator half is
generative like GPT and is scored the same way (BLEU/ROUGE-L), but its training signal — adversarial
reward rather than direct next-token likelihood — is the key mechanical difference that the
activity's comparison matrix is designed to surface.